In [ ]:
from pathlib import Path
import sys

# Make the notebook usable both from the repository root and from a notebooks/tests subfolder.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.psychrometrics.psychrolib_adapter import (
    humidity_ratio_from_t_rh,
    relative_humidity_from_t_w,
    dew_point_from_t_rh,
    dew_point_from_t_w,
    moist_air_enthalpy_from_t_rh,
    moist_air_enthalpy_from_t_w,
    moist_air_density_from_t_rh,
    moist_air_density_from_t_w,
    saturation_humidity_ratio,
)

from core.psychrometrics.moist_air import (
    MoistAirState,
    moist_air_state_from_t_rh,
    moist_air_state_from_t_w,
    saturated_moist_air_state,
)

In [ ]:
T = 293.15
RH = 0.50
p = 101325.0

In [ ]:
# Low-level PsychroLib adapter checks.
W = humidity_ratio_from_t_rh(T, RH, p)
RH_back = relative_humidity_from_t_w(T, W, p)
T_dew_from_rh = dew_point_from_t_rh(T, RH)
T_dew_from_w = dew_point_from_t_w(T, W, p)
h_from_rh = moist_air_enthalpy_from_t_rh(T, RH, p)
h_from_w = moist_air_enthalpy_from_t_w(T, W)
rho_from_rh = moist_air_density_from_t_rh(T, RH, p)
rho_from_w = moist_air_density_from_t_w(T, W, p)
W_sat = saturation_humidity_ratio(T, p)

In [ ]:
# High-level moist-air state helpers.
air = moist_air_state_from_t_rh(T=T, RH=RH, p=p)
air_from_w = moist_air_state_from_t_w(T=T, W=air.W, p=p)
sat = saturated_moist_air_state(T=T, p=p)

assert isinstance(air, MoistAirState)
assert isinstance(air_from_w, MoistAirState)
assert isinstance(sat, MoistAirState)

# Expected order of magnitude at 20 degC, RH=50%, p=101325 Pa.
# Enthalpy must be around 38-39 kJ/kg_dry_air, i.e. 38000-39000 J/kg_dry_air.
# If this test returns around 38,000,000 J/kg_dry_air, the old erroneous x1000 scaling is still present.

assert 0.0070 < W < 0.0076, W
assert abs(RH_back - RH) < 1e-9, (RH_back, RH)
assert 281.5 < T_dew_from_rh < 283.2, T_dew_from_rh
assert abs(T_dew_from_w - T_dew_from_rh) < 1e-6, (T_dew_from_w, T_dew_from_rh)
assert 38000.0 < h_from_rh < 39000.0, h_from_rh
assert abs(h_from_w - h_from_rh) < 1e-6, (h_from_w, h_from_rh)
assert 1.19 < rho_from_rh < 1.21, rho_from_rh
assert abs(rho_from_w - rho_from_rh) < 1e-9, (rho_from_w, rho_from_rh)
assert 0.0142 < W_sat < 0.0152, W_sat

assert abs(air.W - W) < 1e-12
assert abs(air.RH - RH) < 1e-12
assert abs(air.h - h_from_rh) < 1e-9
assert abs(air.rho - rho_from_rh) < 1e-12
assert abs(air.T_dew - T_dew_from_rh) < 1e-9

assert abs(air_from_w.RH - RH) < 1e-9
assert sat.RH > 0.99
assert sat.W > air.W

print("Moist air psychrometrics smoke test passed.")

In [ ]:
{
    "W_kg_per_kg_dry_air": W,
    "RH_back": RH_back,
    "T_dew_K": T_dew_from_rh,
    "h_J_per_kg_dry_air": h_from_rh,
    "h_kJ_per_kg_dry_air_display": h_from_rh / 1000.0,
    "rho_kg_moist_air_per_m3": rho_from_rh,
    "W_sat_kg_per_kg_dry_air": W_sat,
    "air_state": air,
    "saturated_air_state": sat,
}

In [ ]:
from core.psychrometrics.condensation import (
    CondensationOnsetResult,
    check_condensation_onset,
)

air = moist_air_state_from_t_rh(T=293.15, RH=0.50, p=101325.0)

dry_surface = check_condensation_onset(
    air=air,
    T_surface=285.15,
)

wet_surface = check_condensation_onset(
    air=air,
    T_surface=280.15,
)

assert isinstance(dry_surface, CondensationOnsetResult)
assert isinstance(wet_surface, CondensationOnsetResult)

assert dry_surface.will_condense is False
assert dry_surface.dew_point_margin < 0.0
assert len(dry_surface.warnings) == 0

assert wet_surface.will_condense is True
assert wet_surface.dew_point_margin > 0.0
assert wet_surface.W_surface_sat < wet_surface.W_bulk
assert any(w.code == "CONDENSATION_ONSET" for w in wet_surface.warnings)

print("Condensation onset smoke test passed.")

In [ ]:
from core.properties import (
    ConstantPropertyProvider,
    FluidTransportProperties,
)

water_like = FluidTransportProperties(
    rho=998.0,
    mu=1.0e-3,
    k=0.6,
    cp=4180.0,
)

provider = ConstantPropertyProvider(water_like)

props = provider.at(T=293.15, p=101325.0)

assert props.rho == 998.0
assert props.mu == 1.0e-3
assert props.k == 0.6
assert props.cp == 4180.0

print("Minimal property provider smoke test passed.")

In [ ]:
from core.psychrometrics.condensation import (
    CondensationOnsetResult,
    check_condensation_onset,
)

air = moist_air_state_from_t_rh(T=293.15, RH=0.50, p=101325.0)

dry_surface = check_condensation_onset(
    air=air,
    T_surface=285.15,
)

wet_surface = check_condensation_onset(
    air=air,
    T_surface=280.15,
)

frost_surface = check_condensation_onset(
    air=air,
    T_surface=270.15,
)

assert isinstance(dry_surface, CondensationOnsetResult)
assert isinstance(wet_surface, CondensationOnsetResult)
assert isinstance(frost_surface, CondensationOnsetResult)

assert dry_surface.will_condense is False
assert dry_surface.dew_point_margin < 0.0
assert len(dry_surface.warnings) == 0

assert wet_surface.will_condense is True
assert wet_surface.dew_point_margin > 0.0
assert wet_surface.W_surface_sat < wet_surface.W_bulk
assert any(w.code == "CONDENSATION_ONSET" for w in wet_surface.warnings)

assert frost_surface.will_condense is True
assert any(w.code == "CONDENSATION_ONSET" for w in frost_surface.warnings)
assert any(w.code == "SURFACE_BELOW_FREEZING" for w in frost_surface.warnings)

print("Condensation onset smoke test passed.")